# VELOCE III
# Reconstructing Radial Velocity Curves of Classical Cepheids

Giordano Viviani and Richard I. Anderson (2025)

email: giordano.viviani@proton.me

This notebook reproduce step-by-step the analysis conducted to create the framework published in *Viviani et Anderson 2025* (doi: 10.48550/arXiv.2511.10534)

In [1]:
import os
import sys
from pathlib import Path

sys.path.append("../src")
import numpy as np
import pandas as pd

# Import the veloce framework library
import velocefw as vf

## Prepare Data
### VELOCE DR1 data
Accessing the VELOCE DR1 (Anderson, Viviani et al, 2024) data.

The code makes use of the environment variable DATASETS, and it assumes the VELOCE DR1 fits files are in a sub-folder.
Please change the path to where you saved the VELOCE DR1 fits files. 

In [2]:
datasets_dir = Path(os.environ.get("DATASETS", "")).expanduser()
veloce_data_dir = datasets_dir / "veloce_dr1_zenodo"

Reading all the headers into a pandas.DataFrame

In [7]:
# Entries not needed for the analysis
entries_to_remove = ["SIMPLE", "BITPIX", "NAXIS", "EXTEND"]

header_list = []
for f in veloce_data_dir.iterdir():
    if f.suffix.lower() == ".fits":
        header = vf.get_header(f)
        for entry in entries_to_remove:
            header.pop(entry, None)
        header_list.append(header)

veloce_headers = pd.DataFrame(header_list).sort_values("NAME").reset_index(drop=True)

Selecting only Classical Cepheids that have been modeled

In [8]:
veloce_headers = veloce_headers[veloce_headers["BOOL_FIT_AVAILABLE"]]

Get the actual observations for all the targets in with available model.

In [9]:
read_columns = ["BJD", "RV", "RV_ERR"]
veloce_data = pd.DataFrame()
for name in veloce_headers["NAME"]:
    file = veloce_data_dir / f"{name.replace(' ', '_')}.fits"
    temp = vf.get_table(file, 1, columns=read_columns)
    temp["NAME"] = name
    veloce_data = pd.concat([veloce_data, temp], ignore_index=True)

In [10]:
veloce_data

,RV,RV_ERR,BJD,NAME
0,32.185816,0.04900,57329.748702,AA Gem
1,37.898000,0.07200,57330.742757,AA Gem
2,37.576816,0.09600,57331.741786,AA Gem
3,26.085447,0.08500,57332.730356,AA Gem
4,17.024631,0.05700,57333.733664,AA Gem
...,...,...,...,...
17156,-5.809840,0.00255,57478.514867,zet Gem
17157,19.654300,0.00290,59603.675083,zet Gem
17158,20.899990,0.00329,59604.656486,zet Gem
17159,-0.793100,0.00290,59608.678606,zet Gem
